# 02 — ResNet18: clasificación de presencia de D40

Este notebook **no entrena**: lee los resultados generados por

```powershell
python -m src.models.train_resnet --config configs/resnet/exp1_scratch.yaml
python -m src.models.train_resnet --config configs/resnet/exp2_pretrained.yaml
python -m src.models.train_resnet --config configs/resnet/exp3_augment.yaml
python -m src.models.train_resnet --config configs/resnet/exp4_auxiliary.yaml
```

`exp4_auxiliary` agrega D00, D10 y D20 como **tareas auxiliares**: la red las predice, pero la selección del
checkpoint, el threshold y las métricas principales siguen siendo solo de D40.

Cada experimento guarda en `reports/experiments/resnet/<experiment_id>/` su `config.yaml`, `history.csv`,
`metrics.json` y `predictions.csv`. El checkpoint se elige por PR-AUC en `val` y el threshold maximiza F1 en `val`;
ambos quedan fijos para `test_id`, `test_ood` (Norway) y `test_perspective` (China_Drone).

La tarea es **clasificación a nivel de imagen** (`has_D40`), no detección de objetos.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS = PROJECT_ROOT / "reports" / "experiments" / "resnet"
SPLITS = ["val", "test_id", "test_ood", "test_perspective"]

runs = {}
for folder in sorted(RESULTS.glob("*")) if RESULTS.exists() else []:
    if (folder / "metrics.json").exists() and "_limit" not in folder.name:
        runs[folder.name] = dict(
            config=yaml.safe_load((folder / "config.yaml").read_text(encoding="utf-8")),
            metrics=json.loads((folder / "metrics.json").read_text(encoding="utf-8")),
            history=pd.read_csv(folder / "history.csv"),
        )
print("Experimentos encontrados:", list(runs) or "ninguno todavía")

## 1. Configuración registrada

In [ ]:
keys = ["pretrained", "augment", "aux_classes", "aux_weight", "lr", "batch_size", "epochs", "best_epoch", "optimizer", "scheduler",
        "pos_weight", "seed", "dataset_version", "splits_sha256"]
pd.DataFrame({name: {k: run["config"].get(k) for k in keys} for name, run in runs.items()})

## 2. Curvas de entrenamiento

In [ ]:
if runs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, run in runs.items():
        h = run["history"]
        axes[0].plot(h.epoch, h.train_loss, label=f"{name} train")
        axes[0].plot(h.epoch, h.val_loss, "--", label=f"{name} val")
        axes[1].plot(h.epoch, h.val_pr_auc, label=name)
    axes[0].set(title="Loss (BCE con pos_weight)", xlabel="epoch")
    axes[1].set(title="PR-AUC en validation", xlabel="epoch")
    for ax in axes:
        ax.legend(fontsize=8)
    plt.tight_layout()

## 3. Métricas por split

Accuracy no se reporta: con ~9.6 % de prevalencia sería engañosa.

In [ ]:
rows = []
for name, run in runs.items():
    for split, m in run["metrics"]["splits"].items():
        rows.append(dict(experiment=name, split=split, n=m["n"], prevalence=m["prevalence"],
                         precision=m["precision"], recall=m["recall"], f1=m["f1"], pr_auc=m["pr_auc"]))
table = pd.DataFrame(rows)
if not table.empty:
    display(table.pivot(index="experiment", columns="split", values=["f1", "pr_auc"])
            .reindex(columns=SPLITS, level=1).round(4))
table.round(4)

## 4. Matrices de confusión (threshold fijado en val)

In [ ]:
for name, run in runs.items():
    fig, axes = plt.subplots(1, len(SPLITS), figsize=(4 * len(SPLITS), 3.4))
    for ax, split in zip(axes, SPLITS):
        c = run["metrics"]["splits"][split]["confusion_matrix"]
        grid = [[c["tn"], c["fp"]], [c["fn"], c["tp"]]]
        ax.imshow(grid, cmap="Blues")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, grid[i][j], ha="center", va="center")
        ax.set(title=split, xticks=[0, 1], yticks=[0, 1], xticklabels=["No D40", "D40"],
               yticklabels=["No D40", "D40"], xlabel="predicción", ylabel="real")
    fig.suptitle(f"{name} — threshold {run['metrics']['threshold']:.3f}")
    plt.tight_layout()

## 5. Domain shift

Caída de desempeño de `test_id` hacia `test_ood` (Norway, país no visto) y `test_perspective` (China_Drone, vista aérea).
La prevalencia de D40 difiere entre splits, así que la PR-AUC debe leerse junto a la prevalencia (PR-AUC esperada de un clasificador aleatorio).

In [ ]:
if not table.empty:
    order = ["test_id", "test_ood", "test_perspective"]
    shift = table[table.split.isin(order)]
    ax = shift.pivot(index="split", columns="experiment", values="pr_auc").reindex(order).plot.bar(figsize=(8, 4), rot=0)
    baseline = shift.groupby("split").prevalence.first().reindex(order)
    ax.plot(range(3), baseline.values, "k_", markersize=40, label="prevalencia (azar)")
    ax.set(title="PR-AUC por dominio", ylabel="PR-AUC")
    ax.legend(fontsize=8)

## 6. Tareas auxiliares (exp4)

PR-AUC de las grietas que el experimento 4 predice como apoyo. No intervienen en ninguna decisión del modelo;
la pregunta principal es si `exp4_auxiliary` mejora la PR-AUC de **D40** frente a `exp2_pretrained` (tabla de la sección 3).

In [ ]:
aux_rows = [dict(experiment=name, split=split, damage=damage, **values)
            for name, run in runs.items()
            for split, m in run["metrics"]["splits"].items()
            for damage, values in m.get("auxiliary", {}).items()]
pd.DataFrame(aux_rows).pivot(index=["experiment", "damage"], columns="split", values="pr_auc").reindex(
    columns=SPLITS).round(4) if aux_rows else "Sin experimentos con tareas auxiliares todavía"

## 7. Latencia (batch 1, solo forward)

In [ ]:
pd.DataFrame({name: run["metrics"]["latency"] for name, run in runs.items()}).T